# exp092 worst-well visible-test guard

Run the target-free guard against mounted exp092 Kaggle train and inference outputs on the exposed visible test. This normal notebook run does not observe the Code Competition hidden LB test.

## Contents

1. Setup and mounted input check
2. Run visible-test guard
3. Guard outputs

## 1. Setup and mounted input check

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import json
import pandas as pd

from settings import ExperimentPaths, get_nested, load_config
from worst_well_rawtest_guard import OUTPUT_PREFIX, resolve_inputs, run_guard


def cfg_get(config, dotted_key, default=None):
    value = get_nested(config, dotted_key)
    return default if value is None else value


In [ ]:
paths = ExperimentPaths()
paths.ensure_output_dirs()
config = load_config()
guard_config = cfg_get(config, "audit.worst_well_rawtest_guard", {})

print("Experiment:", config["experiment"]["name"])
print("Route:", config["experiment"]["route"])
print("Guard mode:", guard_config.get("mode"))
print("Guard kernel sources:", cfg_get(config, "runtime.kaggle.guard_kernel_sources"))
print("Bootstrap files:", cfg_get(config, "runtime.kaggle.bootstrap_files"))
print("Test data dir:", paths.test_data_dir)
print("Output artifacts dir:", paths.artifacts_dir)


## 2. Run visible-test guard

In [ ]:
guard_output_dir = paths.artifacts_dir / "worst_well_rawtest_guard"
args = SimpleNamespace(
    exp092_predictions=None,
    exp073_predictions=None,
    exp077_submission=None,
    train_feature_schema=None,
    inference_feature_schema=None,
    train_projection_summary=None,
    inference_projection_summary=None,
    oof_by_well="artifacts/oof_delta_guard/exp092_oof_delta_guard_by_well.csv",
    oof_path_continuity="artifacts/oof_delta_guard/exp092_oof_delta_guard_path_continuity.csv",
)
inputs = resolve_inputs(args)
print(json.dumps({key: str(value) for key, value in inputs.items()}, indent=2))

summary = run_guard(
    output_dir=guard_output_dir,
    test_dir=paths.test_data_dir,
    inputs=inputs,
    regression_threshold=float(
        guard_config.get("regression_threshold_vs_exp077_rmse", 0.25)
    ),
    top_n_worst=int(guard_config.get("top_n_worst_wells", 30)),
)


## 3. Guard outputs

In [ ]:
summary_path = guard_output_dir / f"{OUTPUT_PREFIX}_summary.json"
print("Summary:", summary_path)
print("Status:", summary.get("status"))
print("Warning wells:", summary.get("test_warning", {}).get("warning_wells"))

well_metrics = pd.read_csv(guard_output_dir / f"{OUTPUT_PREFIX}_test_well_metrics.csv")
bucket_metrics = pd.read_csv(guard_output_dir / f"{OUTPUT_PREFIX}_test_bucket_metrics.csv")
display(well_metrics.head(20))
display(bucket_metrics)
print("Generated files:")
for path in sorted(guard_output_dir.glob("*")):
    print(path.name, path.stat().st_size)
